# Busca Profunda de Hiperparâmetros — Modelo 3

Continuação do Notebook 1: pega o **modelo vencedor**, faz **busca profunda com Optuna**, define **threshold** via OOF, avalia **calibração**, treina o **modelo final** e gera arquivos/figuras em `OUTPUT_DIR`.


In [1]:
%pip install -U scikit-learn optuna xgboost catboost openpyxl joblib matplotlib shap

Note: you may need to restart the kernel to use updated packages.


In [2]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline

In [3]:

import os, json, hashlib, subprocess
from copy import deepcopy
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold
)
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    balanced_accuracy_score, matthews_corrcoef,
    brier_score_loss, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.utils import check_random_state

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None

try:
    from sklearn.frozen import FrozenEstimator
except Exception:
    FrozenEstimator = None


# CONFIG

BASES_DIR   = "bases"
BASE_FILE   = "baseModelo3-FatoresAssociadosAoObito.xlsx"
TARGET_COL  = "OBITO"

TEST_SIZE    = 0.20
RANDOM_STATE = 42

INNER_SPLITS = 10

N_ITER_DEEP      = 300
N_JOBS           = -1
OPTUNA_N_STARTUP = 30

ES_ROUNDS_SEARCH = 50
ES_MARGIN        = 1.10

OPTIMIZE_METRIC = "average_precision"

THRESH_METRIC = "precision"
MIN_RECALL    = 0.8

RS_BAYES = RANDOM_STATE + 999
RS_OOF   = RANDOM_STATE + 777
RS_CALIB = RANDOM_STATE + 555

OOF_SPLITS  = 10
OOF_REPEATS = 3

EVALUATE_CALIBRATION = True
CALIBRATION_METHODS  = ["sigmoid", "isotonic"]
CALIBRATION_CV       = 5
CALIBRATION_REPEATS  = 2

ES_VALIDATION_FRACTION = 0.10
ES_ROUNDS_FINAL        = 50

SENSITIVITY_THRESHOLDS = {
    "F1-ótimo":        {"metric": "f1",        "min_recall": None},
    "Youden":          {"metric": "youden",    "min_recall": None},
    "Prec@R>=0.5":     {"metric": "precision", "min_recall": 0.5},
    "Prec@R>=0.7":     {"metric": "precision", "min_recall": 0.7},
    "Prec@R>=0.8":     {"metric": "precision", "min_recall": 0.8},
}

DEMOGRAPHIC_CANDIDATES = ["REG_NOTIF", "IDADE", "SEXO", "AUTOCTONE", "PERI_SAZONAL"]

N_BOOTSTRAPS = 2000
CI_LEVEL     = 0.95

OUTPUT_DIR = "resultados_final_base3_sem_hosp"
SAVE_MODEL = True

NB1_JSON_PATH = "resultados_selecao_modelo_base3/resultados_selecao_modelo_modelo3.json"

os.makedirs(OUTPUT_DIR, exist_ok=True)

/home/vkusterl/Documents/ICs/Bitka/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

# Funções auxiliares

METRIC_NAMES = ["roc_auc", "average_precision", "f1", "precision", "recall", "balanced_accuracy", "mcc", "brier"]

def safe_clone_estimator(est):
    try:
        return clone(est)
    except RuntimeError as e:
        msg = str(e)
        if "cat_features" not in msg:
            raise

        if isinstance(est, Pipeline):
            new_steps = [(name, safe_clone_estimator(step)) for name, step in est.steps]
            return Pipeline(steps=new_steps, memory=est.memory, verbose=est.verbose)

        params = est.get_params(deep=False)
        return est.__class__(**params)

def compute_file_hash(filepath, algorithm="sha256"):
    h = hashlib.new(algorithm)
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def get_pip_freeze():
    try:
        r = subprocess.run(["pip", "freeze"], capture_output=True, text=True, timeout=30)
        return r.stdout.strip()
    except Exception as e:
        return f"pip freeze falhou: {e}"

def get_proba(model, X):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        return p[:, 1] if p.ndim == 2 else p.ravel()
    s = model.decision_function(X)
    return 1 / (1 + np.exp(-s))

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

class ProbCalibrator:
    """Calibrador 1D picklable para probabilidades (p -> p_cal).
    - method="none": identidade
    - method="isotonic": IsotonicRegression
    - method="sigmoid": Platt scaling via LogisticRegression em logit(p)
    """
    def __init__(self, method, iso=None, lr=None, eps=1e-6):
        self.method = method
        self.iso = iso
        self.lr = lr
        self.eps = float(eps)

    @classmethod
    def fit(cls, method, y, p, eps=1e-6):
        y = np.asarray(y)
        p = np.asarray(p, dtype=float)

        if method == "none":
            return cls(method="none", eps=eps)

        if method == "isotonic":
            iso = IsotonicRegression(out_of_bounds="clip")
            iso.fit(p, y)
            return cls(method="isotonic", iso=iso, eps=eps)

        if method == "sigmoid":
            p_clip = np.clip(p, eps, 1 - eps)
            Xc = np.log(p_clip / (1 - p_clip)).reshape(-1, 1)

            lr = LogisticRegression(solver="lbfgs", max_iter=1000)
            lr.fit(Xc, y)
            return cls(method="sigmoid", lr=lr, eps=eps)

        raise ValueError(f"Método de calibração desconhecido: {method}")

    def transform(self, p):
        p = np.asarray(p, dtype=float)

        if self.method == "none":
            return p

        if self.method == "isotonic":
            return self.iso.transform(p)

        if self.method == "sigmoid":
            eps = self.eps
            p = np.clip(p, eps, 1 - eps)
            X = np.log(p / (1 - p)).reshape(-1, 1)
            return self.lr.predict_proba(X)[:, 1]

        raise ValueError(f"Método de calibração desconhecido: {self.method}")

    def __call__(self, p):
        return self.transform(p)

def fit_prob_calibrator(method, y, p):
    """Mantido por compatibilidade: agora retorna um ProbCalibrator (picklable)."""
    return ProbCalibrator.fit(method, y, p)

class ProbaCalibratedWrapper:
    """Wrapper que aplica um calibrador picklable sobre predict_proba do estimador base."""
    def __init__(self, base_estimator, calibrator):
        self.base_estimator = base_estimator
        self.calibrator = calibrator

    def predict_proba(self, X):
        p = get_proba(self.base_estimator, X)
        p_cal = self.calibrator(p)
        return np.column_stack([1.0 - p_cal, p_cal])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

def compute_metrics(y_true, y_proba, y_pred):
    return {
        "roc_auc":           float(roc_auc_score(y_true, y_proba)) if len(np.unique(y_true)) > 1 else float("nan"),
        "average_precision": float(average_precision_score(y_true, y_proba)),
        "f1":                float(f1_score(y_true, y_pred, zero_division=0)),
        "precision":         float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":            float(recall_score(y_true, y_pred, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "mcc":               float(matthews_corrcoef(y_true, y_pred)),
        "brier":             float(brier_score_loss(y_true, y_proba)),
    }

def select_threshold(y_true, proba, metric="f1", min_recall=None):
    thresholds = np.linspace(0.001, 0.999, 999)
    best_t, best_key = None, None

    for t in thresholds:
        y_pred = (proba >= t).astype(int)
        rec = recall_score(y_true, y_pred, zero_division=0)
        if min_recall is not None and rec < min_recall:
            continue

        prec = precision_score(y_true, y_pred, zero_division=0)
        f1   = f1_score(y_true, y_pred, zero_division=0)

        if metric == "precision":
            val = prec
        elif metric == "f1":
            val = f1
        elif metric == "youden":
            tn = np.sum((y_true == 0) & (y_pred == 0))
            fp = np.sum((y_true == 0) & (y_pred == 1))
            spec = tn / (tn + fp + 1e-12)
            val = rec + spec - 1
        else:
            raise ValueError("metric deve ser 'precision', 'f1' ou 'youden'")

        key = (val, rec, t)
        if best_key is None or key > best_key:
            best_key = key
            best_t = float(t)

    if best_t is None:
        raise ValueError(f"Nenhum threshold atingiu min_recall={min_recall}.")
    return best_t, float(best_key[0])

def bootstrap_ci(y_true, y_proba, y_pred, n_boot=2000, ci=0.95, random_state=42):
    rng = check_random_state(random_state)
    y_true  = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    y_pred  = np.asarray(y_pred)

    idx0 = np.where(y_true == 0)[0]
    idx1 = np.where(y_true == 1)[0]

    stats = {k: [] for k in METRIC_NAMES}
    for _ in range(n_boot):
        s = np.concatenate([
            rng.choice(idx0, len(idx0), replace=True),
            rng.choice(idx1, len(idx1), replace=True),
        ])
        m = compute_metrics(y_true[s], y_proba[s], y_pred[s])
        for k, v in m.items():
            stats[k].append(v)

    alpha = (1 - ci) / 2
    out = {}
    for k, arr in stats.items():
        arr = np.asarray(arr, dtype=float)
        out[k] = {
            "mean_boot": float(np.nanmean(arr)),
            "ci_low":    float(np.nanquantile(arr, alpha)),
            "ci_high":   float(np.nanquantile(arr, 1 - alpha)),
        }
    return out

def threshold_sensitivity_analysis(y_true, proba, threshold_configs, holdout_y=None, holdout_proba=None):
    rows = []
    for name, cfg in threshold_configs.items():
        try:
            t, _ = select_threshold(y_true, proba, cfg["metric"], cfg["min_recall"])
            y_pred = (proba >= t).astype(int)
            row = {"Critério": name, "Threshold": float(t)}
            row.update({f"OOF_{k}": v for k, v in compute_metrics(y_true, proba, y_pred).items()})

            if holdout_y is not None and holdout_proba is not None:
                y_pred_h = (holdout_proba >= t).astype(int)
                row.update({f"Holdout_{k}": v for k, v in compute_metrics(holdout_y, holdout_proba, y_pred_h).items()})
            rows.append(row)
        except Exception as e:
            rows.append({"Critério": name, "Threshold": None, "Nota": str(e)})
    return pd.DataFrame(rows)

class FinalModelWithThreshold(BaseEstimator, ClassifierMixin):
    def __init__(self, model, threshold, model_name="unknown", metadata=None):
        self.model = model
        self.threshold = float(threshold)
        self.model_name = str(model_name)
        self.metadata = metadata or {}

    def fit(self, X, y, **fit_params):
        self.model.fit(X, y, **fit_params)
        return self

    def predict_proba(self, X):
        return self.model.predict_proba(X)

    def predict(self, X):
        return (get_proba(self.model, X) >= self.threshold).astype(int)

    def predict_with_threshold(self, X, threshold=None):
        t = self.threshold if threshold is None else float(threshold)
        p = get_proba(self.model, X)
        return (p >= t).astype(int), p

# CatBoost + sklearn (algumas versões de sklearn reclamam do "tags")
from sklearn.base import ClassifierMixin
class SafeCatBoostClassifier(ClassifierMixin, CatBoostClassifier):
    pass

class NominalPreprocessor(BaseEstimator):
    # imputa e tipa: num -> float; cat -> string ou category
    def __init__(self, num_cols, cat_cols, cat_as="string", cat_categories=None,
                 num_strategy="median", cat_strategy="most_frequent", cat_missing_token="MISSING"):
        self.num_cols = num_cols
        self.cat_cols = cat_cols
        self.cat_as = cat_as
        self.cat_categories = cat_categories
        self.num_strategy = num_strategy
        self.cat_strategy = cat_strategy
        self.cat_missing_token = cat_missing_token

    def fit(self, X, y=None):
        X = X.copy()
        self.num_cols_ = list(self.num_cols)
        self.cat_cols_ = list(self.cat_cols)

        self._num_imp = SimpleImputer(strategy=self.num_strategy).fit(X[self.num_cols_])
        if self.cat_strategy == "constant":
            self._cat_imp = SimpleImputer(strategy="constant", fill_value=self.cat_missing_token).fit(X[self.cat_cols_].astype("object"))
        else:
            self._cat_imp = SimpleImputer(strategy=self.cat_strategy).fit(X[self.cat_cols_].astype("object"))

        self._cat_categories_ = None
        if self.cat_categories is not None:
            self._cat_categories_ = {c: [str(v) for v in self.cat_categories[c]] for c in self.cat_cols_ if c in self.cat_categories}
        return self

    def transform(self, X):
        Xo = X.copy()
        Xo[self.num_cols_] = self._num_imp.transform(Xo[self.num_cols_]).astype(float)

        cat = self._cat_imp.transform(Xo[self.cat_cols_].astype("object"))
        cat_df = pd.DataFrame(cat, columns=self.cat_cols_, index=Xo.index).astype(str)

        if self._cat_categories_:
            for c in self.cat_cols_:
                if c in self._cat_categories_:
                    cat_df[c] = pd.Categorical(cat_df[c], categories=self._cat_categories_[c])

        if self.cat_as == "category":
            for c in self.cat_cols_:
                cat_df[c] = cat_df[c].astype("category")

        Xo[self.cat_cols_] = cat_df
        return Xo

def build_preprocessor(model_name, num_cols, cat_cols, cat_categories):
    model_name = str(model_name).lower()
    if model_name == "catboost":
        return NominalPreprocessor(num_cols, cat_cols, cat_as="string", cat_categories=cat_categories)
    if model_name == "xgboost":
        return NominalPreprocessor(num_cols, cat_cols, cat_as="category", cat_categories=cat_categories)

    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), list(num_cols)),
            ("cat", Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), list(cat_cols)),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

def make_pipeline(estimator, model_name, num_cols, cat_cols, cat_categories):
    try:
        est = safe_clone_estimator(estimator)
    except Exception:
        est = deepcopy(estimator)

    model_name = str(model_name).lower()
    if model_name == "xgboost":
        try:
            est.set_params(enable_categorical=True, tree_method="hist")
        except Exception:
            pass
    if model_name == "catboost":
        try:
            est.set_params(cat_features=list(cat_cols))
        except Exception:
            pass

    return Pipeline([("prep", build_preprocessor(model_name, num_cols, cat_cols, cat_categories)), ("model", est)])

# def safe_clone_estimator(est):
#     try:
#         return clone(est)
#     except RuntimeError as e:
#         msg = str(e)
#         # CatBoost conflita com sklearn>=1.4 (cat_features)
#         if "modifies parameter cat_features" not in msg and "parameter cat_features" not in msg:
#             raise

#         # Se for Pipeline, reconstrói recursivamente os steps
#         if isinstance(est, Pipeline):
#             new_steps = [(name, safe_clone_estimator(step)) for name, step in est.steps]
#             return Pipeline(steps=new_steps, memory=est.memory, verbose=est.verbose)

#         # Caso base (ex: CatBoostClassifier): reinstancia direto pelos params
#         params = est.get_params(deep=False)
#         return est.__class__(**params)

def compute_oof_predictions(pipeline, X_data, y_data, cv, random_state, fit_params=None):
    n = len(y_data)
    oof_sum   = np.zeros(n, dtype=float)
    oof_count = np.zeros(n, dtype=int)

    safe_fit_params = None
    if fit_params:
        safe_fit_params = {k: np.asarray(v) for k, v in fit_params.items()}

    for i, (tr_idx, te_idx) in enumerate(cv.split(X_data, y_data)):
        est = safe_clone_estimator(pipeline)
        try:
            est.set_params(**{"model__random_state": random_state + i})
        except Exception:
            try:
                est.set_params(**{"model__random_seed": random_state + i})
            except Exception:
                pass

        fp = {}
        if safe_fit_params:
            fp = {k: v[tr_idx] for k, v in safe_fit_params.items()}

        est.fit(X_data.iloc[tr_idx], y_data[tr_idx], **fp)
        oof_sum[te_idx]   += get_proba(est, X_data.iloc[te_idx])
        oof_count[te_idx] += 1

    return oof_sum / oof_count

def feature_names_from_pipeline(pipe, X_ref):
    prep = pipe.named_steps.get("prep", None)
    if prep is not None and hasattr(prep, "get_feature_names_out"):
        return list(prep.get_feature_names_out())
    return list(X_ref.columns)


In [5]:

# Dados

base_path = os.path.join(BASES_DIR, BASE_FILE)
if not os.path.exists(base_path):
    if os.path.exists(BASE_FILE):
        base_path = BASE_FILE
    elif os.path.exists(os.path.join("/mnt/data", BASE_FILE)):
        base_path = os.path.join("/mnt/data", BASE_FILE)

df = pd.read_excel(base_path)
df = df.drop(columns="HOSPITALIZACAO")
print(df.columns.tolist())
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].values

print("Base:", BASE_FILE, "| shape:", X.shape, "| pos:", int(y.sum()), "| neg:", int((y==0).sum()))
data_hash = compute_file_hash(base_path)
print("SHA-256:", data_hash[:16] + "...")

NUM_COLS = ["IDADE", "DIAS_ATE_NOTIF"]
for c in NUM_COLS:
    if c not in X.columns:
        raise ValueError(f"Coluna numérica esperada não encontrada: {c}")
CAT_COLS = [c for c in X.columns if c not in NUM_COLS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

n_pos_train = int(y_train.sum())
n_neg_train = int((y_train==0).sum())
CLASS_RATIO_TRAIN = n_neg_train / max(n_pos_train, 1)

print("Treino:", X_train.shape, "| ratio neg/pos:", f"{CLASS_RATIO_TRAIN:.2f}")
print("Teste :", X_test.shape)

CAT_CATEGORIES = {c: [str(v) for v in sorted(pd.Series(X_train[c]).dropna().unique().tolist(), key=lambda x: str(x))] for c in CAT_COLS}

sw_train = np.where(y_train == 1, CLASS_RATIO_TRAIN, 1.0)
sw_train = sw_train / sw_train.mean()

nb1_results = None
if os.path.exists(NB1_JSON_PATH):
    with open(NB1_JSON_PATH, "r", encoding="utf-8") as f:
        nb1_results = json.load(f)
    assert X_train.index.tolist() == nb1_results["holdout_split"]["train_index"]
    assert X_test.index.tolist()  == nb1_results["holdout_split"]["test_index"]
    print("Split validado com Notebook 1.")
else:
    print("JSON do Notebook 1 não encontrado em", NB1_JSON_PATH)

detected_demographics = [c for c in X_train.columns if c in DEMOGRAPHIC_CANDIDATES]
print("Demográficas detectadas:", detected_demographics if detected_demographics else "nenhuma")


['REG_NOTIF', 'IDADE', 'SEXO', 'VACINA_FA', 'SINT_HEMORRAGICO', 'DISTUR_RENAL', 'SINAL_FAGET', 'DOR_ABDOMINAL', 'RESULT_IGM', 'RESULT_PCR', 'HISTOPATOLOGIA', 'IMUNOHISTOQUIMICA', 'R_ISOVIRAL', 'CRIT_CONF', 'AUTOCTONE', 'PERI_SAZONAL', 'DIAS_ATE_NOTIF', 'OBITO', 'INTERNACAO']
Base: baseModelo3-FatoresAssociadosAoObito.xlsx | shape: (17147, 18) | pos: 2076 | neg: 15071
SHA-256: d452d06824d10708...
Treino: (13717, 18) | ratio neg/pos: 7.26
Teste : (3430, 18)
Split validado com Notebook 1.
Demográficas detectadas: ['REG_NOTIF', 'IDADE', 'SEXO', 'AUTOCTONE', 'PERI_SAZONAL']


In [6]:

# Modelo vencedor

if nb1_results is not None:
    winner_model = nb1_results["winner_model"]
else:
    winner_model = "catboost"  # fallback

print("Modelo vencedor:", winner_model)


Modelo vencedor: catboost


In [ ]:

# OPTUNA (busca profunda)

def build_optuna_search_specs():
    specs = {}

    specs["decision_tree"] = {
        "suggest_params": lambda trial: {
            "max_depth":             trial.suggest_int("max_depth", 1, 40),
            "min_samples_split":     trial.suggest_int("min_samples_split", 2, 80),
            "min_samples_leaf":      trial.suggest_int("min_samples_leaf", 1, 40),
            "criterion":             trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "max_leaf_nodes":        trial.suggest_int("max_leaf_nodes", 10, 500),
            "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0.0, 0.05),
        },
        "build_estimator": lambda params, rs, cr: DecisionTreeClassifier(class_weight="balanced", random_state=rs, **params),
        "uses_early_stopping": False,
        "uses_sample_weight":  False,
    }

    specs["random_forest"] = {
        "suggest_params": lambda trial: {
            "n_estimators":          trial.suggest_int("n_estimators", 200, 2000),
            "max_depth":             trial.suggest_int("max_depth", 2, 50),
            "min_samples_split":     trial.suggest_int("min_samples_split", 2, 60),
            "min_samples_leaf":      trial.suggest_int("min_samples_leaf", 1, 30),
            "max_features":          trial.suggest_float("max_features", 0.1, 1.0),
            "criterion":             trial.suggest_categorical("criterion", ["gini", "entropy"]),
            "min_impurity_decrease": trial.suggest_float("min_impurity_decrease", 0.0, 0.02),
        },
        "build_estimator": lambda params, rs, cr: RandomForestClassifier(
            class_weight="balanced", bootstrap=True, n_jobs=1, random_state=rs, **params
        ),
        "uses_early_stopping": False,
        "uses_sample_weight":  False,
    }

    specs["adaboost"] = {
        "suggest_params": lambda trial: {
            "n_estimators":   trial.suggest_int("n_estimators", 50, 1000),
            "learning_rate":  trial.suggest_float("learning_rate", 1e-3, 2.0, log=True),
            "base_max_depth": trial.suggest_int("base_max_depth", 1, 5),
        },
        "build_estimator": lambda params, rs, cr: AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=params.pop("base_max_depth"), random_state=rs),
            n_estimators=params.pop("n_estimators"),
            learning_rate=params.pop("learning_rate"),
            random_state=rs,
        ),
        "uses_early_stopping": False,
        "uses_sample_weight":  True,
    }

    specs["xgboost"] = {
        "suggest_params": lambda trial: {
            "max_depth":        trial.suggest_int("max_depth", 2, 12),
            "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 30.0, log=True),
            "gamma":            trial.suggest_float("gamma", 0.0, 10.0),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-3, 200.0, log=True),
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-4, 50.0, log=True),
        },
        "build_estimator": lambda params, rs, cr: XGBClassifier(
            objective="binary:logistic", eval_metric="aucpr",
            tree_method="hist", enable_categorical=True,
            n_estimators=10000, early_stopping_rounds=ES_ROUNDS_SEARCH,
            n_jobs=1, random_state=rs,
            scale_pos_weight=cr, **params
        ),
        "uses_early_stopping": True,
        "uses_sample_weight":  False,
    }

    specs["catboost"] = {
        "suggest_params": lambda trial: {
            "depth":            trial.suggest_int("depth", 3, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
            "l2_leaf_reg":      trial.suggest_float("l2_leaf_reg", 1e-3, 100.0, log=True),
            "random_strength":  trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "border_count":     trial.suggest_int("border_count", 32, 255),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
        },
        "build_estimator": lambda params, rs, cr: SafeCatBoostClassifier(
            loss_function="Logloss", eval_metric="PRAUC",
            auto_class_weights="Balanced",
            iterations=10000,
            od_type="Iter", od_wait=ES_ROUNDS_SEARCH,
            random_state=rs, verbose=False,
            allow_writing_files=False, thread_count=1,
            cat_features=list(CAT_COLS),
            **params,
        ),
        "uses_early_stopping": True,
        "uses_sample_weight":  False,
    }

    return specs


deep_specs = build_optuna_search_specs()
spec = deep_specs[winner_model]

inner_cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=RS_BAYES)

def create_objective(spec, X_data, y_data, cv, optimize_metric, class_ratio, random_state, model_name, sw=None):
    uses_es = spec["uses_early_stopping"]
    uses_sw = spec["uses_sample_weight"]

    def objective(trial):
        params = spec["suggest_params"](trial)
        fold_scores = []
        fold_best = []

        for fold_idx, (tr_idx, val_idx) in enumerate(cv.split(X_data, y_data)):
            X_tr, X_val = X_data.iloc[tr_idx], X_data.iloc[val_idx]
            y_tr, y_val = y_data[tr_idx], y_data[val_idx]

            est = spec["build_estimator"](dict(params), random_state + fold_idx, class_ratio)

            if uses_es:
                prep = build_preprocessor(model_name, NUM_COLS, CAT_COLS, CAT_CATEGORIES)
                prep.fit(X_tr)
                X_tr_p = prep.transform(X_tr)
                X_val_p = prep.transform(X_val)

                if isinstance(est, XGBClassifier):
                    est.fit(X_tr_p, y_tr, eval_set=[(X_val_p, y_val)], verbose=False)
                    best_iter = est.best_iteration
                else:
                    est.fit(X_tr_p, y_tr, eval_set=(X_val_p, y_val), verbose=False)
                    best_iter = est.get_best_iteration()

                if best_iter is not None:
                    fold_best.append(best_iter)

                proba_val = est.predict_proba(X_val_p)[:, 1]                    
                proba_tr  = est.predict_proba(X_tr_p)[:, 1] if optimize_metric == "f1" else None  

            else:
                pipe = make_pipeline(est, model_name, NUM_COLS, CAT_COLS, CAT_CATEGORIES)
                fp = {}
                if uses_sw and sw is not None:
                    fp["model__sample_weight"] = np.asarray(sw)[tr_idx]
                pipe.fit(X_tr, y_tr, **fp)

                proba_val = get_proba(pipe, X_val)                             
                proba_tr  = get_proba(pipe, X_tr) if optimize_metric == "f1" else None  

            if optimize_metric == "average_precision":
                score = average_precision_score(y_val, proba_val)          
            elif optimize_metric == "roc_auc":
                score = roc_auc_score(y_val, proba_val)                 
            elif optimize_metric == "f1":
                t_star, _ = select_threshold(y_tr, proba_tr, metric="f1", min_recall=None)
                y_pred_val = (proba_val >= t_star).astype(int)
                score = f1_score(y_val, y_pred_val, zero_division=0)
            else:
                raise ValueError("métrica não suportada")

            fold_scores.append(score)
            trial.report(score, fold_idx)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        if fold_best:
            trial.set_user_attr("median_n_best", int(np.median(fold_best)))
            trial.set_user_attr("all_n_best", fold_best)

        return float(np.mean(fold_scores))

    return objective

sampler = TPESampler(seed=RS_BAYES, n_startup_trials=OPTUNA_N_STARTUP)
pruner  = MedianPruner(n_startup_trials=10, n_warmup_steps=2)

study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner, study_name=f"deep_{winner_model}")
objective_fn = create_objective(
    spec=spec, X_data=X_train, y_data=y_train, cv=inner_cv,
    optimize_metric=OPTIMIZE_METRIC, class_ratio=CLASS_RATIO_TRAIN,
    random_state=RANDOM_STATE, model_name=winner_model,
    sw=sw_train if spec["uses_sample_weight"] else None,
)

study.optimize(objective_fn, n_trials=N_ITER_DEEP, n_jobs=N_JOBS, show_progress_bar=True)

best_trial  = study.best_trial
best_params = best_trial.params
best_score  = float(best_trial.value)

n_best_cv = best_trial.user_attrs.get("median_n_best", None)
all_n_best = best_trial.user_attrs.get("all_n_best", None)

print("Best", OPTIMIZE_METRIC, ":", f"{best_score:.4f}")
print("Params:", best_params)
print("n_best_cv:", n_best_cv)

# plot rápido: convergência + distribuição de n_best (se existir)
best_values = []
cur_best = -1e9
for t in study.trials:
    if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None:
        cur_best = max(cur_best, t.value)
    best_values.append(cur_best)

plt.figure(figsize=(7.5, 4.2))
plt.plot(best_values)
plt.title("Optuna - melhor score por trial")
plt.xlabel("trial")
plt.ylabel(OPTIMIZE_METRIC)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "convergence_curve.png"), dpi=150, bbox_inches="tight")
plt.show()

if all_n_best:
    plt.figure(figsize=(7.0, 4.2))
    plt.hist(all_n_best, bins=20)
    plt.title("Distribuição de n_best (best_iteration) no melhor trial")
    plt.xlabel("n_best")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "nbest_distribution.png"), dpi=150, bbox_inches="tight")
    plt.show()

# pipeline base (para OOF / calibração). Se ES, fixa iterações no n_best_cv.
best_est = spec["build_estimator"](dict(best_params), RANDOM_STATE, CLASS_RATIO_TRAIN)

if spec["uses_early_stopping"] and n_best_cv is not None:
    if isinstance(best_est, XGBClassifier):
        best_est.set_params(n_estimators=n_best_cv, early_stopping_rounds=None)
    else:
        best_est.set_params(iterations=n_best_cv, use_best_model=False)

best_pipeline = make_pipeline(best_est, winner_model, NUM_COLS, CAT_COLS, CAT_CATEGORIES)


[I 2026-02-25 13:07:35,481] A new study created in memory with name: deep_catboost
  0%|          | 0/300 [00:00<?, ?it/s]

In [ ]:

# THRESHOLD (OOF)

oof_cv = RepeatedStratifiedKFold(n_splits=OOF_SPLITS, n_repeats=OOF_REPEATS, random_state=RS_OOF)

fit_params = {}
if spec["uses_sample_weight"]:
    fit_params["model__sample_weight"] = sw_train

proba_oof = compute_oof_predictions(best_pipeline, X_train, y_train, oof_cv, RS_OOF, fit_params=fit_params)

t_star, t_val = select_threshold(y_train, proba_oof, THRESH_METRIC, MIN_RECALL)
pred_oof = (proba_oof >= t_star).astype(int)
oof_metrics = compute_metrics(y_train, proba_oof, pred_oof)

print("Threshold OOF:", f"{t_star:.3f}", "|", THRESH_METRIC, "=", f"{t_val:.4f}")
print("OOF metrics:", oof_metrics)

sens_df = threshold_sensitivity_analysis(y_train, proba_oof, SENSITIVITY_THRESHOLDS)
sens_path = os.path.join(OUTPUT_DIR, "threshold_sensitivity_oof.csv")
sens_df.to_csv(sens_path, index=False, encoding="utf-8")
print("Tabela:", sens_path)


In [ ]:

# CALIBRAÇÃO (seleção)

DO_CALIBRATION = False
CHOSEN_CAL_METHOD = "none"

def _make_prefit_calibrator(trained_estimator, method):
    if FrozenEstimator is not None:
        return CalibratedClassifierCV(estimator=FrozenEstimator(trained_estimator), method=method, ensemble=False)
    return CalibratedClassifierCV(estimator=trained_estimator, method=method, cv="prefit")

def _safe_scores(y_true, proba):
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return None
    return {
        "brier":  brier_score_loss(y_true, proba),
        "pr_auc": average_precision_score(y_true, proba),
        "roc_auc": roc_auc_score(y_true, proba),
    }

def _agg(records):
    out = {}
    for k in ["brier", "pr_auc", "roc_auc"]:
        vals = [r[k] for r in records if r is not None]
        out[k] = float(np.mean(vals)) if vals else None
    return out

if not EVALUATE_CALIBRATION:
    print("Calibração desativada.")
else:
    cal_cv = RepeatedStratifiedKFold(n_splits=CALIBRATION_CV, n_repeats=CALIBRATION_REPEATS, random_state=RS_CALIB)

    base_records = []
    for i, (tr_idx, val_idx) in enumerate(cal_cv.split(X_train, y_train)):
        est_base = safe_clone_estimator(best_pipeline)
        fp = {}
        if spec["uses_sample_weight"]:
            fp["model__sample_weight"] = np.asarray(sw_train)[tr_idx]
        est_base.fit(X_train.iloc[tr_idx], y_train[tr_idx], **fp)

        # cross-fit simples: divide val em 2 dobras
        skf2 = StratifiedKFold(n_splits=2, shuffle=True, random_state=RS_CALIB + i)
        for a_idx, b_idx in skf2.split(X_train.iloc[val_idx], y_train[val_idx]):
            va = val_idx[a_idx]
            vb = val_idx[b_idx]
            p = get_proba(est_base, X_train.iloc[vb])
            sc = _safe_scores(y_train[vb], p)
            if sc: base_records.append(sc)

    base_res = _agg(base_records)
    print("Base (none):", base_res)

    best_cal_method = "none"
    best_brier = base_res["brier"]
    base_prauc = base_res["pr_auc"]

    for method in CALIBRATION_METHODS:
        cal_records = []
        for i, (tr_idx, val_idx) in enumerate(cal_cv.split(X_train, y_train)):
            est_base = safe_clone_estimator(best_pipeline)
            fp = {}
            if spec["uses_sample_weight"]:
                fp["model__sample_weight"] = np.asarray(sw_train)[tr_idx]
            est_base.fit(X_train.iloc[tr_idx], y_train[tr_idx], **fp)

            skf2 = StratifiedKFold(n_splits=2, shuffle=True, random_state=RS_CALIB + 100 + i)
            for a_idx, b_idx in skf2.split(X_train.iloc[val_idx], y_train[val_idx]):
                va = val_idx[a_idx]
                vb = val_idx[b_idx]

                cal = _make_prefit_calibrator(est_base, method)
                cal.fit(X_train.iloc[va], y_train[va])

                p = get_proba(cal, X_train.iloc[vb])
                sc = _safe_scores(y_train[vb], p)
                if sc: cal_records.append(sc)

        res = _agg(cal_records)
        print("Method", method, ":", res)

        if res["brier"] is not None and res["pr_auc"] is not None:
            if (res["brier"] < best_brier) and (res["pr_auc"] >= base_prauc * 0.995):
                best_brier = res["brier"]
                best_cal_method = method

    DO_CALIBRATION = best_cal_method != "none"
    CHOSEN_CAL_METHOD = best_cal_method
    print("Escolha:", CHOSEN_CAL_METHOD, "| aplicar:", DO_CALIBRATION)


In [ ]:

# TREINO FINAL + HOLDOUT 

# treino base (com early stopping "real" quando aplicável)
uses_es = spec["uses_early_stopping"]
es_applied = False
n_best_final = None

if uses_es and n_best_cv is not None:
    n_best_final = int(np.ceil(n_best_cv * ES_MARGIN))

# treinar um modelo base final (sem calibração)
if uses_es and n_best_final is not None:
    n_train = len(y_train)
    rng = check_random_state(RANDOM_STATE)
    idx = np.arange(n_train)

    # split interno p/ early stopping
    sss = train_test_split(idx, test_size=ES_VALIDATION_FRACTION, stratify=y_train, random_state=RANDOM_STATE)
    tr_idx, va_idx = sss[0], sss[1]

    est_es = spec["build_estimator"](dict(best_params), RANDOM_STATE, CLASS_RATIO_TRAIN)

    if isinstance(est_es, XGBClassifier):
        est_es.set_params(n_estimators=n_best_final, early_stopping_rounds=ES_ROUNDS_FINAL)
    else:
        est_es.set_params(iterations=n_best_final, od_wait=ES_ROUNDS_FINAL, od_type="Iter")

    prep = build_preprocessor(winner_model, NUM_COLS, CAT_COLS, CAT_CATEGORIES)
    prep.fit(X_train.iloc[tr_idx])

    X_tr_p = prep.transform(X_train.iloc[tr_idx])
    X_va_p = prep.transform(X_train.iloc[va_idx])

    if spec["uses_sample_weight"]:
        sw_tr = np.asarray(sw_train)[tr_idx]
    else:
        sw_tr = None

    fit_args = {}
    if sw_tr is not None and isinstance(est_es, XGBClassifier):
        # xgboost sample_weight direto
        fit_args["sample_weight"] = sw_tr

    if isinstance(est_es, XGBClassifier):
        est_es.fit(X_tr_p, y_train[tr_idx], eval_set=[(X_va_p, y_train[va_idx])], verbose=False, **fit_args)
    else:
        est_es.fit(X_tr_p, y_train[tr_idx], eval_set=(X_va_p, y_train[va_idx]), verbose=False)

    best_iter = getattr(est_es, "best_iteration", None)
    if best_iter is None and hasattr(est_es, "get_best_iteration"):
        best_iter = est_es.get_best_iteration()
    if best_iter is not None:
        n_best_final = int(best_iter)
    es_applied = True

    final_model = Pipeline([("prep", prep), ("model", est_es)])
else:
    est_no_es = spec["build_estimator"](dict(best_params), RANDOM_STATE, CLASS_RATIO_TRAIN)
    final_model = make_pipeline(est_no_es, winner_model, NUM_COLS, CAT_COLS, CAT_CATEGORIES)
    fp = {}
    if spec["uses_sample_weight"]:
        fp["model__sample_weight"] = sw_train
    final_model.fit(X_train, y_train, **fp)

# calibração (se recomendada) e threshold recalculado em OOF calibrado
final_estimator = final_model

if DO_CALIBRATION:
    # Calibração aprendida em OOF (sem leakage) e aplicada ao modelo final treinado em todo X_train
    cal_fn = fit_prob_calibrator(CHOSEN_CAL_METHOD, y_train, proba_oof)
    proba_cal_oof = cal_fn(proba_oof)
    t_star, t_val = select_threshold(y_train, proba_cal_oof, THRESH_METRIC, MIN_RECALL)
    final_estimator = ProbaCalibratedWrapper(final_model, cal_fn)

print("Threshold final:", f"{t_star:.3f}", "| calib:", CHOSEN_CAL_METHOD)

proba_holdout = get_proba(final_estimator, X_test)
pred_holdout  = (proba_holdout >= t_star).astype(int)
holdout_metrics = compute_metrics(y_test, proba_holdout, pred_holdout)

print("Holdout metrics:", holdout_metrics)

# sensibilidade no holdout com os thresholds do OOF
sens_holdout_df = threshold_sensitivity_analysis(
    y_train,
    (proba_cal_oof if DO_CALIBRATION else proba_oof),
    SENSITIVITY_THRESHOLDS,
    holdout_y=y_test,
    holdout_proba=proba_holdout
)
sens_holdout_path = os.path.join(OUTPUT_DIR, "threshold_sensitivity_holdout.csv")
sens_holdout_df.to_csv(sens_holdout_path, index=False, encoding="utf-8")
print("Tabela:", sens_holdout_path)


In [ ]:

# PLOTS BÁSICOS 

# Confusion matrix
cm = confusion_matrix(y_test, pred_holdout)
disp = ConfusionMatrixDisplay(cm)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
disp.plot(ax=ax, colorbar=False)
ax.set_title("Matriz de Confusão (holdout)")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

# ROC + PR
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.6))
RocCurveDisplay.from_predictions(y_test, proba_holdout, ax=axes[0])
axes[0].set_title("ROC")

PrecisionRecallDisplay.from_predictions(y_test, proba_holdout, ax=axes[1])
axes[1].set_title("Precision-Recall")

fig.suptitle("Curvas (holdout)")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "roc_pr_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

# Calibration curve
frac_pos, mean_pred = calibration_curve(y_test, proba_holdout, n_bins=10)
plt.figure(figsize=(6.0, 4.2))
plt.plot(mean_pred, frac_pos, marker="o")
plt.plot([0,1],[0,1], linestyle="--")
plt.title("Curva de calibração (holdout)")
plt.xlabel("Probabilidade prevista")
plt.ylabel("Fração positiva")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "calibration_curve.png"), dpi=150, bbox_inches="tight")
plt.show()

# Learning curve (simples: score vs tamanho do treino)
try:
    from sklearn.model_selection import learning_curve
    sizes, train_scores, val_scores = learning_curve(
        estimator=safe_clone_estimator(best_pipeline),
        X=X_train, y=y_train,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        scoring="average_precision",
        train_sizes=np.linspace(0.2, 1.0, 5),
        n_jobs=N_JOBS,
    )
    plt.figure(figsize=(6.5, 4.3))
    plt.plot(sizes, np.mean(train_scores, axis=1), marker="o", label="treino")
    plt.plot(sizes, np.mean(val_scores, axis=1), marker="o", label="cv")
    plt.title("Learning curve (PR-AUC)")
    plt.xlabel("n amostras")
    plt.ylabel("PR-AUC")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "learning_curve.png"), dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print("Learning curve falhou:", e)

# Importância: nativa (se existir) + permutation
try:
    model_for_imp = final_estimator
    if hasattr(model_for_imp, "calibrated_classifiers_"):
        model_for_imp = model_for_imp.calibrated_classifiers_[0].estimator

    if hasattr(model_for_imp, "named_steps"):
        # pipeline
        X_imp = model_for_imp.named_steps["prep"].transform(X_test)
        names = feature_names_from_pipeline(model_for_imp, X_test)
        mdl = model_for_imp.named_steps["model"]
    else:
        X_imp, names, mdl = X_test, list(X_test.columns), model_for_imp

    if hasattr(mdl, "feature_importances_"):
        imp = np.asarray(mdl.feature_importances_, dtype=float)
        idx = np.argsort(-imp)[:25]
        plt.figure(figsize=(8.0, 5.2))
        plt.barh(np.array(names)[idx][::-1], imp[idx][::-1])
        plt.title("Importância (nativa)")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "feature_importance_native.png"), dpi=150, bbox_inches="tight")
        plt.show()

    # permutation importance (no pipeline)
    perm = permutation_importance(final_estimator, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring="average_precision")
    idx = np.argsort(-perm.importances_mean)[:25]
    plt.figure(figsize=(8.0, 5.2))
    plt.barh(np.array(X_test.columns)[idx][::-1], perm.importances_mean[idx][::-1])
    plt.title("Importância (permutation)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "feature_importance_permutation.png"), dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print("Importância falhou:", e)


In [ ]:

# SHAP

import shap

model_for_shap = final_estimator

# Desembrulha calibração:
# - nosso wrapper OOF (ProbaCalibratedWrapper) => .base_estimator
# - CalibratedClassifierCV do sklearn (se existir) => .calibrated_classifiers_[0].estimator
while True:
    if hasattr(model_for_shap, "base_estimator"):
        model_for_shap = model_for_shap.base_estimator
        continue
    if hasattr(model_for_shap, "calibrated_classifiers_"):
        model_for_shap = model_for_shap.calibrated_classifiers_[0].estimator
        continue
    break

X_for_model = X_test
feature_names = list(X_test.columns)

if hasattr(model_for_shap, "named_steps") and "prep" in model_for_shap.named_steps:
    prep = model_for_shap.named_steps["prep"]
    X_for_model = prep.transform(X_test)
    if not isinstance(X_for_model, pd.DataFrame):
        feature_names = feature_names_from_pipeline(model_for_shap, X_test)
        X_for_model = pd.DataFrame(X_for_model, columns=feature_names, index=X_test.index)
    model_for_shap = model_for_shap.named_steps["model"]

explainer = shap.TreeExplainer(model_for_shap)
shap_vals = explainer.shap_values(X_for_model)
if isinstance(shap_vals, list):
    shap_vals = shap_vals[1]

plt.figure(figsize=(9, 5))
shap.summary_plot(shap_vals, X_for_model, show=False)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_summary.png"), dpi=150, bbox_inches="tight")
plt.show()

mean_abs = np.abs(shap_vals).mean(axis=0)
idx = np.argsort(-mean_abs)[:25]
plt.figure(figsize=(8.0, 5.2))
plt.barh(np.array(feature_names)[idx][::-1], mean_abs[idx][::-1])
plt.title("SHAP mean(|value|)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_importance.png"), dpi=150, bbox_inches="tight")
plt.show()



In [ ]:

# SUBGRUPOS + EXTRAS

# Subgrupos (métricas por categoria)
subgroup_results = {}
if detected_demographics:
    for col in detected_demographics:
        subgroup_results[col] = {}
        for v in sorted(pd.Series(X_test[col]).dropna().unique().tolist(), key=lambda x: str(x)):
            mask = (X_test[col] == v).values
            if mask.sum() < 30:
                continue
            m = compute_metrics(y_test[mask], proba_holdout[mask], pred_holdout[mask])
            subgroup_results[col][str(v)] = {"n": int(mask.sum()), **m}

    with open(os.path.join(OUTPUT_DIR, "subgroup_analysis.json"), "w", encoding="utf-8") as f:
        json.dump(subgroup_results, f, ensure_ascii=False, indent=2, default=str)
    print("subgroup_analysis.json salvo.")
else:
    print("Sem variáveis demográficas para subgrupos.")

# Métricas saúde + CM normalizada
tn, fp, fn, tp = confusion_matrix(y_test, pred_holdout).ravel()
specificity = tn / (tn + fp + 1e-12)
npv = tn / (tn + fn + 1e-12)
youden = holdout_metrics["recall"] + specificity - 1
metrics_saude = {
    **holdout_metrics,
    "specificity": float(specificity),
    "npv": float(npv),
    "youden": float(youden),
    "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
}
pd.DataFrame([metrics_saude]).to_csv(os.path.join(OUTPUT_DIR, "holdout_metrics_saude.csv"), index=False, encoding="utf-8")

cmn = confusion_matrix(y_test, pred_holdout, normalize="true")
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(cmn).plot(ax=ax, colorbar=False, values_format=".2f")
ax.set_title("Matriz de Confusão (normalizada)")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix_normalized.png"), dpi=150, bbox_inches="tight")
plt.show()

# certeza (probabilidade da classe escolhida)
certainty = np.where(pred_holdout == 1, proba_holdout, 1 - proba_holdout)
certainty_df = pd.DataFrame({
    "index": X_test.index,
    "y_true": y_test,
    "proba_pos": proba_holdout,
    "y_pred": pred_holdout,
    "certeza_pred": certainty,
})
certainty_df.to_csv(os.path.join(OUTPUT_DIR, "holdout_predicoes_com_certeza.csv"), index=False, encoding="utf-8")

plt.figure(figsize=(6.5, 4.2))
plt.hist(certainty, bins=30)
plt.title("Histograma de certeza (holdout)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pred_certainty_hist.png"), dpi=150, bbox_inches="tight")
plt.show()

# cobertura vs acurácia (corta pelos mais "certos")
qs = np.linspace(0.1, 1.0, 10)
cov = []
acc = []
for q in qs:
    thr = np.quantile(certainty, 1-q)
    m = certainty >= thr
    cov.append(float(m.mean()))
    acc.append(float((pred_holdout[m] == y_test[m]).mean()) if m.sum()>0 else np.nan)

plt.figure(figsize=(6.5, 4.2))
plt.plot(cov, acc, marker="o")
plt.title("Cobertura vs Acurácia (holdout)")
plt.xlabel("Cobertura")
plt.ylabel("Acurácia")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "coverage_vs_accuracy.png"), dpi=150, bbox_inches="tight")
plt.show()

# acurácia por bins de certeza
bins = np.linspace(0, 1, 6)
labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]
bin_id = np.digitize(certainty, bins, right=True) - 1
rows = []
for i, lab in enumerate(labels):
    m = bin_id == i
    if m.sum() == 0:
        continue
    rows.append({"bin": lab, "n": int(m.sum()), "accuracy": float((pred_holdout[m]==y_test[m]).mean())})
bin_df = pd.DataFrame(rows)

plt.figure(figsize=(6.5, 4.2))
plt.bar(bin_df["bin"], bin_df["accuracy"])
plt.title("Acurácia por bins de certeza")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "accuracy_vs_certainty_bins.png"), dpi=150, bbox_inches="tight")
plt.show()

# Decision curve
pts = np.linspace(0.01, 0.99, 99)
n = len(y_test)
nb = []
for pt in pts:
    pred_pt = (proba_holdout >= pt).astype(int)
    tp_pt = ((pred_pt==1) & (y_test==1)).sum()
    fp_pt = ((pred_pt==1) & (y_test==0)).sum()
    nb.append(tp_pt/n - fp_pt/n * (pt/(1-pt)))
plt.figure(figsize=(6.5, 4.2))
plt.plot(pts, nb)
plt.title("Decision Curve (net benefit)")
plt.xlabel("threshold")
plt.ylabel("net benefit")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "decision_curve_net_benefit.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:

# SHAP extras (top 3 + locais)

if "shap_vals" not in globals():
    raise RuntimeError("SHAP não foi calculado no bloco anterior.")

n_show = min(1500, len(X_for_model))
X_sh = X_for_model.iloc[:n_show]
S_sh = shap_vals[:n_show]

mean_abs = np.abs(S_sh).mean(axis=0)
top_idx = np.argsort(-mean_abs)[:min(3, S_sh.shape[1])]
top_feats = [X_sh.columns[i] for i in top_idx]

for f in top_feats:
    plt.figure(figsize=(8.5, 5.0))
    shap.dependence_plot(f, S_sh, X_sh, interaction_index="auto", show=False)
    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, f"shap_dependence_{str(f).replace(' ', '_')}.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()

idx_high = int(np.argmax(proba_holdout))
idx_border = int(np.argmin(np.abs(proba_holdout - t_star)))

base_value = explainer.expected_value if np.isscalar(explainer.expected_value) else explainer.expected_value[0]

def _local(idx, tag):
    x_row = X_for_model.iloc[idx]
    s_row = shap_vals[idx]
    exp = shap.Explanation(values=s_row, base_values=base_value, data=x_row.values, feature_names=list(X_for_model.columns))
    plt.figure(figsize=(9, 5.0))
    shap.plots.waterfall(exp, show=False, max_display=15)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"shap_waterfall_{tag}.png"), dpi=150, bbox_inches="tight")
    plt.show()

    try:
        fp = shap.force_plot(base_value, s_row, x_row, feature_names=list(X_for_model.columns), matplotlib=False)
        shap.save_html(os.path.join(OUTPUT_DIR, f"shap_force_{tag}.html"), fp)
    except Exception:
        pass

_local(idx_high, "alto_risco")
_local(idx_border, "limiar")

idx_low = int(np.argmin(proba_holdout))
idx_mid = int(np.argsort(np.abs(proba_holdout - np.median(proba_holdout)))[0])
idxs = list(dict.fromkeys([idx_high, idx_border, idx_mid, idx_low]))

plt.figure(figsize=(10, 5.2))
shap.decision_plot(base_value, shap_vals[idxs, :], X_for_model.iloc[idxs, :], feature_names=list(X_for_model.columns), show=False, feature_order="importance")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_decision_plot.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:

# SALVAR ARTEFATOS

# bootstrap CI
ci = bootstrap_ci(y_test, proba_holdout, pred_holdout, n_boot=N_BOOTSTRAPS, ci=CI_LEVEL, random_state=RANDOM_STATE)
ci_df = pd.DataFrame([{ "metric": k, **v } for k, v in ci.items()])
ci_df.to_csv(os.path.join(OUTPUT_DIR, "bootstrap_ci.csv"), index=False, encoding="utf-8")

# relatório principal
final_payload = {
    "generated_at": datetime.now().isoformat(timespec="seconds"),
    "pipeline": "Busca Profunda de Hiperparâmetros",
    "winner_model": winner_model,
    "config": {
        "base_file": BASE_FILE,
        "target_col": TARGET_COL,
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "inner_splits": INNER_SPLITS,
        "n_iter_deep": N_ITER_DEEP,
        "optimize_metric": OPTIMIZE_METRIC,
        "thresh_metric": THRESH_METRIC,
        "min_recall": MIN_RECALL,
        "n_best_cv": n_best_cv,
        "n_best_final": n_best_final,
        "early_stopping_in_search": bool(spec["uses_early_stopping"]),
        "early_stopping_applied_final": bool(es_applied),
    },
    "reproducibility": {"data_sha256": data_hash},
    "best_params": best_params,
    "threshold": {"t_star": t_star, "criterion_value": t_val},
    "calibration": {"evaluate": EVALUATE_CALIBRATION, "chosen_method": CHOSEN_CAL_METHOD, "applied": DO_CALIBRATION},
    "oof_metrics": oof_metrics,
    "holdout_metrics": holdout_metrics,
    "bootstrap_ci": ci,
}

with open(os.path.join(OUTPUT_DIR, "final_report.json"), "w", encoding="utf-8") as f:
    json.dump(final_payload, f, ensure_ascii=False, indent=2, default=str)

# relatório estendido (só adiciona)
extended_payload = dict(final_payload)
extended_payload["holdout_metrics_saude"] = metrics_saude
with open(os.path.join(OUTPUT_DIR, "final_report_extended.json"), "w", encoding="utf-8") as f:
    json.dump(extended_payload, f, ensure_ascii=False, indent=2, default=str)

# holdout metrics csv
pd.DataFrame([holdout_metrics]).to_csv(os.path.join(OUTPUT_DIR, "holdout_metrics.csv"), index=False, encoding="utf-8")

# split indices (para reproduzir holdout)
split_info = {
    "train_index": X_train.index.tolist(),
    "test_index": X_test.index.tolist(),
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
}
with open(os.path.join(OUTPUT_DIR, "holdout_split_indices.json"), "w", encoding="utf-8") as f:
    json.dump(split_info, f, ensure_ascii=False, indent=2)

# model metadata
model_meta = {
    "winner_model": winner_model,
    "threshold": t_star,
    "data_sha256": data_hash,
    "chosen_calibration_method": CHOSEN_CAL_METHOD,
    "applied_calibration": DO_CALIBRATION,
}
with open(os.path.join(OUTPUT_DIR, "model_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(model_meta, f, ensure_ascii=False, indent=2)

# pip freeze
with open(os.path.join(OUTPUT_DIR, "pip_freeze.txt"), "w") as f:
    f.write(get_pip_freeze())

# salvar modelo
if SAVE_MODEL:
    from joblib import dump
    model_path = os.path.join(OUTPUT_DIR, f"final_model_{winner_model}.joblib")
    dump(final_estimator, model_path)

    wrapped = FinalModelWithThreshold(final_estimator, t_star, model_name=winner_model, metadata=model_meta)
    wrapper_path = os.path.join(OUTPUT_DIR, f"final_model_wrapped_{winner_model}.joblib")
    dump(wrapped, wrapper_path)

print("Saída em:", OUTPUT_DIR)
